# Testing SU(2) Symmetry of the Slater Determinant Reference State

This notebook verifies that the Slater determinant reference state selected using the `su2_symmetry=true` and `use_slater_reference=true` options in our load functions is a perfect SU(2) singlet state (i.e. $\langle S^2 \rangle = 0$).

In [ ]:
using Lattices
using LinearAlgebra
using SparseArrays
using JLD2
using HDF5
using Test

# Include project code
if !isdefined(Main, :UtilityFunctions)
    include("utility_functions.jl")
end
using .UtilityFunctions
include("ed_objects.jl")
include("ed_functions.jl")

## 1. Load JLD2 Data with SU(2) Symmetry & Slater Reference

We load a $3\times2$ grid dataset with $N = (3,3)$ spin-singlet sector.

In [ ]:
folder = "data/N=(3, 3)_3x2"
U_values, target_vecs, indexer, precomputed_structures, N, spin_conserved, use_symmetry, sign_convention = 
    load_ED_data(folder; verbose=true, su2_symmetry=true, use_slater_reference=true)

println("Loaded sector details:")
println("Electron Count N: ", N)
println("JW Sign Convention: ", sign_convention)
println("target_vecs dimensions: ", size(target_vecs))

## 2. Extract Reference State & Build $S^2$ Operator

We pull the reference state (row 1) and construct the $S^2$ operator in the subspace basis.

In [ ]:
# Row 1 of target_vecs is the Slater determinant reference state when use_slater_reference=true
psi_ref = Vector{ComplexF64}(target_vecs[1, :])
dim = length(indexer.inv_comb_dict)

println("Reference state norm: ", norm(psi_ref))
println("Hilbert space dimension: ", dim)

# Build S² operator using the indexer
rows = Int[]
cols = Int[]
vals = Float64[]
create_S2!(rows, cols, vals, 1.0, indexer; momentum_basis=true, sign_convention=sign_convention)
S2_op = sparse(rows, cols, vals, dim, dim)
println("Constructed S² operator (sparse size: ", size(S2_op), ") with ", nnz(S2_op), " non-zero elements.")

## 3. Measure $S^2$ Expectation Value

We compute the expectation value $\langle \psi_{\text{ref}} | S^2 | \psi_{\text{ref}} \rangle$.

In [ ]:
s2_val = real(dot(psi_ref, S2_op * psi_ref))

println("Measured <S²> expectation value: ", s2_val)
println("Target S² for singlet state (S=0): 0.0")

if abs(s2_val) < 1e-12
    println("SUCCESS: The Slater determinant reference state is a perfect SU(2) singlet (S=0)!")
else
    println("FAILURE: Expected 0.0, got ", s2_val)
end

## 4. Verify with HDF5 Data

We verify the same behavior on a $3\times3$ HDF5 dataset with $N = (3,3)$.

In [ ]:
h5_folder = "data_new_sign/N=(3, 3)_3x3"
U_values_h5, target_vecs_h5, indexer_h5, _, N_h5, _, _, sign_convention_h5 = 
    load_ED_data(h5_folder; verbose=true, su2_symmetry=true, use_slater_reference=true)

psi_ref_h5 = Vector{ComplexF64}(target_vecs_h5[1, :])
dim_h5 = length(indexer_h5.inv_comb_dict)

rows_h5 = Int[]
cols_h5 = Int[]
vals_h5 = Float64[]
create_S2!(rows_h5, cols_h5, vals_h5, 1.0, indexer_h5; momentum_basis=true, sign_convention=sign_convention_h5)
S2_op_h5 = sparse(rows_h5, cols_h5, vals_h5, dim_h5, dim_h5)

s2_val_h5 = real(dot(psi_ref_h5, S2_op_h5 * psi_ref_h5))
println("H5 Measured <S²> expectation value: ", s2_val_h5)
if abs(s2_val_h5) < 1e-12
    println("SUCCESS: The H5 Slater determinant reference state is a perfect SU(2) singlet (S=0)!")
else
    println("FAILURE: Expected 0.0, got ", s2_val_h5)
end